# convT-as-flipped-padded-conv — ex2: hand-checked 2x2 input / 2x2 kernel ConvT equivalence

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `convT-as-flipped-padded-conv`. Running the final beacon cell reports progress against the `CNN: ConvT as flipped padded conv` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: ConvT as flipped padded conv` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`convT-as-flipped-padded-conv`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "convT-as-flipped-padded-conv"
DD_SUBTOPIC = "CNN: ConvT as flipped padded conv"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## ConvT-as-flipped-padded-conv — quick refresher

A stride-1 `F.conv_transpose2d(x, w)` is exactly equivalent to `F.conv2d(pad(x, K-1), flip_swap(w))`, where:

- `pad(x, K-1)` zero-pads the spatial axes by `K-1` on every side.
- `flip_swap(w) = w.flip([2, 3]).transpose(0, 1).contiguous()` spatially flips the kernel and swaps the channel axes.

**This drill (ex2) vs ex1.** ex1 implemented the equivalence in full generality (large random `x` / `w`, tolerance-checked against `F.conv_transpose2d`). ex2 reduces the question to a single **hand-checked** numeric example — a `2×2` input and a `2×2` kernel whose expected `3×3` output can be computed by hand. The exercise forces you to confront the per-cell arithmetic and verify *byte-exact* equality (no `atol`).

### Exercise 2 — hand-checked 2x2 input / 2x2 kernel ConvT equivalence

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze the stride-1 ConvT-as-flipped-padded-conv equivalence on a hand-computable `2×2` input / `2×2` identity-diagonal kernel, demonstrating byte-exact agreement between the rebuilt conv2d path and the canonical ConvT2d output.
> Keywords: hand-check, byte-exact, convT, flip-and-pad
> ```

**KCs targeted:** `convT-padded-conv-equivalence`, `convT-kernel-flip-rule`

Implement `ex2_hand_check_convT_equivalence()`.

You will build a tiny example and verify the equivalence is exact (no tolerance). The setup is fixed:

- Input `x` of shape `(1, 1, 2, 2)` with values `[[1., 2.], [3., 4.]]`.
- Weight `w` of shape `(1, 1, 2, 2)` (ConvT2d layout `(IC, OC, KH, KW)`) with values `[[1., 0.], [0., 1.]]` — a diagonal 2x2 kernel.

The expected stride-1 `F.conv_transpose2d(x, w)` output (computable by hand from the four overlapping kernel placements) is the `(1, 1, 3, 3)` tensor:

```
[[1., 2., 0.],
 [3., 5., 2.],
 [0., 3., 4.]]
```

**Steps.**
1. Build `x` and `w` (the fixed values above).
2. Compute `convT_out = F.conv_transpose2d(x, w)`.
3. Compute the equivalent via conv2d: pad `x` by `K-1=1` on every spatial side, flip `w` spatially via `w.flip([2, 3])`, swap channel axes via `.transpose(0, 1).contiguous()`, then run `F.conv2d(x_pad, w_flipped_swapped)`.
4. Verify both equal the hand-computed expected matrix above EXACTLY (use `t.equal`, not `t.allclose`).
5. Return a dict `{'convT': convT_out, 'conv_equiv': conv_out, 'expected': expected}`.

The function takes no arguments — everything is fixed for the hand-check.

The visualization renders the three matrices side by side as heatmaps so you can eyeball the equivalence.

In [ ]:
def ex2_hand_check_convT_equivalence() -> dict:
    import torch.nn.functional as F
    x = t.tensor([[[[1., 2.], [3., 4.]]]])
    w = t.tensor([[[[1., 0.], [0., 1.]]]])   # ConvT layout (IC=1, OC=1, KH=2, KW=2)
    convT_out = F.conv_transpose2d(x, w)
    # Equivalent path: pad K-1=1, flip kernel spatially, swap channel axes.
    x_pad = F.pad(x, (1, 1, 1, 1))
    w_flipped = w.flip([2, 3]).transpose(0, 1).contiguous()
    conv_equiv = F.conv2d(x_pad, w_flipped)
    expected = t.tensor([[[[1., 2., 0.],
                            [3., 5., 2.],
                            [0., 3., 4.]]]])
    return {'convT': convT_out, 'conv_equiv': conv_equiv, 'expected': expected}


<details><summary>Solution</summary>

```python
def ex2_hand_check_convT_equivalence() -> dict:
    import torch.nn.functional as F
    x = t.tensor([[[[1., 2.], [3., 4.]]]])
    w = t.tensor([[[[1., 0.], [0., 1.]]]])   # ConvT layout (IC=1, OC=1, KH=2, KW=2)
    convT_out = F.conv_transpose2d(x, w)
    # Equivalent path: pad K-1=1, flip kernel spatially, swap channel axes.
    x_pad = F.pad(x, (1, 1, 1, 1))
    w_flipped = w.flip([2, 3]).transpose(0, 1).contiguous()
    conv_equiv = F.conv2d(x_pad, w_flipped)
    expected = t.tensor([[[[1., 2., 0.],
                            [3., 5., 2.],
                            [0., 3., 4.]]]])
    return {'convT': convT_out, 'conv_equiv': conv_equiv, 'expected': expected}
```

**Why this example is fully hand-checkable.** With the diagonal kernel `[[1, 0], [0, 1]]`, the ConvT places one copy of the `(1, 2, 3, 4)` input at each of its four valid overlap positions in the `3×3` output grid — the centre cell receives two contributions (`1 + 4 = 5`), the off-diagonal cells one. No floating-point error to absorb, so `t.equal` (byte-exact) replaces `t.allclose`.

**Why `t.equal` matters.** Tolerance-based assertions can hide small layout / sign bugs that only show up at larger scale. The hand-check exposes the equivalence at machine-integer precision — if the byte-exact equality breaks on this tiny case, the larger tolerance-checked equivalence in ex1 is almost certainly broken too, just hidden by the tolerance.

**Difference from ex1.** ex1 reproduced the equivalence on random `(B, IC, H, W)` inputs with random kernels and verified via `t.allclose` — the *generality* check. ex2 reduces to one fixed input that can be computed by hand and verifies via `t.equal` — the *precision* check. Both are needed: random for coverage, hand-checked for catching layout bugs that randomness might average out.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()